# Aula 10 — Análise Integrada com Plotly Interativo

## Objetivos
1. Aplicar **todas as técnicas vistas no curso** numa análise completa.
2. Produzir **dashboards interativos** com Plotly.
3. Escrever um relatório de análise reproduzível.

---

## Roteiro

1. **Pergunta de pesquisa:** Como se comportou o IPCA e o PIB per capita brasileiro
   na última década, e o que os dados nos dizem?
2. Coleta → descritiva → visualização → inferência → modelagem.
3. Dashboards Plotly para apresentação.

---

## 1. Setup

In [ ]:
# === SETUP PARA GOOGLE COLAB ===
# Esta célula garante que o notebook funcione tanto em ambiente local
# quanto em quem clonar o repositório direto no Colab.

import sys, os, subprocess

# 1) Instala dependências (silencioso se já existirem)
pkgs = ["numpy", "pandas", "matplotlib", "seaborn", "scipy", "requests", "plotly", "statsmodels"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# 2) Se estivermos no Colab e o repo ainda não foi clonado, clona.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/220719/curso-estatistica-python.git"  # ajuste após criar o repo

if IN_COLAB and not os.path.exists("/content/curso-estatistica-python"):
    subprocess.run(["git", "clone", REPO_URL, "/content/curso-estatistica-python"], check=False)

# 3) Adiciona o diretório utils ao PYTHONPATH para importar o módulo SIDRA
BASE = "/content/curso-estatistica-python" if IN_COLAB else ".."
if BASE not in sys.path:
    sys.path.append(BASE)

# 4) Pasta de gráficos (cria se não existir)
GRAFICOS_DIR = os.path.join(BASE, "graficos")
os.makedirs(GRAFICOS_DIR, exist_ok=True)

print("✓ Ambiente pronto. Pasta de gráficos:", GRAFICOS_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.2f}".format)

## 2. Coleta integrada

In [ ]:
from utils.sidra_api import obter_ipca_mensal, obter_pib_per_capita_uf, obter_populacao_uf

ipca   = obter_ipca_mensal(120)
pib_uf = obter_pib_per_capita_uf()
pop_uf = obter_populacao_uf(2022)

# Painel UF
uf = pib_uf.merge(pop_uf, on="uf")
uf["log_pop"] = np.log10(uf["populacao"])

print(f"IPCA: {len(ipca)} meses")
print(f"UFs:  {len(uf)} unidades")

## 3. Estatística descritiva consolidada

In [ ]:
resumo = pd.DataFrame({
    "IPCA mensal (%)":      ipca["variacao_mensal"].describe(),
    "PIB per capita (R$)":  uf["pib_per_capita"].describe(),
    "População (mi)":       (uf["populacao"] / 1e6).describe(),
})
resumo

## 4. Dashboard interativo — IPCA com Plotly

In [ ]:
# Acumulado anual (somando log(1+r) para evitar viés multiplicativo)
ipca["ano"] = ipca["data"].dt.year
ipca["log1p_r"] = np.log1p(ipca["variacao_mensal"] / 100)
acumulado = ipca.groupby("ano")["log1p_r"].sum().apply(lambda x: (np.exp(x) - 1) * 100).reset_index()
acumulado.columns = ["ano", "ipca_anual"]

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=("IPCA mensal", "IPCA anual acumulado (%)",
                                    "Distribuição do IPCA mensal", "Volatilidade móvel 12m"),
                    specs=[[{"type": "scatter"}, {"type": "bar"}],
                           [{"type": "histogram"}, {"type": "scatter"}]])

# (1,1) série mensal
fig.add_trace(go.Scatter(x=ipca["data"], y=ipca["variacao_mensal"],
                         mode="lines", name="IPCA mensal",
                         line=dict(color="#1f77b4")), row=1, col=1)

# (1,2) anual acumulado
cores = ["#d62728" if v > 4.5 else "#2ca02c" for v in acumulado["ipca_anual"]]
fig.add_trace(go.Bar(x=acumulado["ano"], y=acumulado["ipca_anual"],
                     marker_color=cores, name="IPCA anual"), row=1, col=2)

# (2,1) histograma
fig.add_trace(go.Histogram(x=ipca["variacao_mensal"], nbinsx=25,
                            marker_color="#ff7f0e", name="Distribuição"), row=2, col=1)

# (2,2) volatilidade móvel
vol = ipca["variacao_mensal"].rolling(12).std()
fig.add_trace(go.Scatter(x=ipca["data"], y=vol, mode="lines",
                          line=dict(color="purple"), name="σ 12m"), row=2, col=2)

fig.update_layout(height=750, showlegend=False,
                   title_text="Dashboard IPCA — Últimos 10 anos",
                   title_font_size=18)
fig.write_html(os.path.join(GRAFICOS_DIR, "aula10_dashboard_ipca.html"))
fig.show()

## 5. Análise inferencial: o IPCA mudou após 2020?

In [ ]:
antes  = ipca[ipca["data"] < "2020-01-01"]["variacao_mensal"]
depois = ipca[ipca["data"] >= "2020-01-01"]["variacao_mensal"]

# Estatísticas
print(f"Antes (n={len(antes)}):  μ = {antes.mean():.3f}%,  σ = {antes.std():.3f}")
print(f"Depois (n={len(depois)}): μ = {depois.mean():.3f}%, σ = {depois.std():.3f}")

# Teste de Welch
t, p = stats.ttest_ind(antes, depois, equal_var=False)
print(f"\nWelch's t = {t:.3f}, p = {p:.4f}")

# IC para a diferença das médias
diff = depois.mean() - antes.mean()
se_diff = np.sqrt(antes.var(ddof=1)/len(antes) + depois.var(ddof=1)/len(depois))
gl_w = (antes.var(ddof=1)/len(antes) + depois.var(ddof=1)/len(depois))**2 / \
       ((antes.var(ddof=1)/len(antes))**2/(len(antes)-1) + (depois.var(ddof=1)/len(depois))**2/(len(depois)-1))
tcrit = stats.t.ppf(0.975, gl_w)
print(f"Diferença (depois − antes): {diff:+.3f} p.p.")
print(f"IC 95%: [{diff - tcrit*se_diff:+.3f}, {diff + tcrit*se_diff:+.3f}] p.p.")

## 6. PIB per capita: análise interativa por UF

In [ ]:
fig = px.scatter(uf, x="populacao", y="pib_per_capita",
                  size="populacao", color="pib_per_capita",
                  hover_name="uf", log_x=True,
                  color_continuous_scale="Viridis",
                  size_max=50,
                  title="PIB per capita vs População por UF (interativo)",
                  labels={"populacao": "População (log)",
                          "pib_per_capita": "PIB per capita (R$)"})
fig.update_layout(height=550)
fig.write_html(os.path.join(GRAFICOS_DIR, "aula10_scatter_uf.html"))
fig.show()

## 7. Mapa coroplético do PIB per capita

In [ ]:
# Códigos IBGE → siglas (mapeamento mínimo necessário para o choropleth)
uf_sigla = {
    "Rondônia":"RO","Acre":"AC","Amazonas":"AM","Roraima":"RR","Pará":"PA",
    "Amapá":"AP","Tocantins":"TO","Maranhão":"MA","Piauí":"PI","Ceará":"CE",
    "Rio Grande do Norte":"RN","Paraíba":"PB","Pernambuco":"PE","Alagoas":"AL",
    "Sergipe":"SE","Bahia":"BA","Minas Gerais":"MG","Espírito Santo":"ES",
    "Rio de Janeiro":"RJ","São Paulo":"SP","Paraná":"PR","Santa Catarina":"SC",
    "Rio Grande do Sul":"RS","Mato Grosso do Sul":"MS","Mato Grosso":"MT",
    "Goiás":"GO","Distrito Federal":"DF"
}
uf["sigla"] = uf["uf"].map(uf_sigla)

# Choropleth usando o geojson público do IBGE
geojson_url = "https://raw.githubusercontent.com/codeforgermany/click_that_hood/main/public/data/brazil-states.geojson"

fig = px.choropleth(uf,
                    geojson=geojson_url,
                    locations="sigla",
                    featureidkey="properties.sigla",
                    color="pib_per_capita",
                    color_continuous_scale="Viridis",
                    hover_name="uf",
                    title="PIB per capita por UF (R$)")
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(height=600)
fig.write_html(os.path.join(GRAFICOS_DIR, "aula10_mapa_pib.html"))
fig.show()

## 8. Modelo final: regressão multivariada simples

In [ ]:
# Mesmo modelo da aula 09, agora com interpretação detalhada
X = sm.add_constant(uf["log_pop"])
y = uf["pib_per_capita"]
modelo = sm.OLS(y, X).fit()

print(modelo.summary())

### Interpretação do modelo

In [ ]:
b0 = modelo.params.iloc[0]
b1 = modelo.params.iloc[1]
r2 = modelo.rsquared

print(f"Equação: PIB_pc ≈ {b0:,.0f} + {b1:,.0f} × log10(População)")
print(f"R² = {r2:.3f} → {r2*100:.1f}% da variância do PIB per capita é explicada")
print(f"\nInterpretação:")
print(f"- Cada vez que a população é multiplicada por 10 (Δ log10 = 1),")
print(f"  o PIB per capita varia, em média, em R$ {b1:+,.0f}.")
print(f"- Sinal {'+' if b1>0 else '-'}: UFs mais populosas tendem a ter PIB per capita "
      f"{'maior' if b1>0 else 'menor'}.")

## 9. Conclusões da análise

In [ ]:
print("""
═══════════════════════════════════════════════════════════
RELATÓRIO DE ANÁLISE — IPCA & PIB PER CAPITA (10 ANOS)
═══════════════════════════════════════════════════════════

1. INFLAÇÃO (IPCA)
   • Média mensal histórica:    {:.3f}%
   • Volatilidade (σ):          {:.3f}%
   • Distribuição: levemente assimétrica à direita, com caudas
     pesadas (eventos extremos mais frequentes que o normal).

2. EFEITO PANDEMIA
   • Pré-2020:  μ = {:.3f}%
   • Pós-2020:  μ = {:.3f}%
   • Diferença estatisticamente {} (Welch p = {:.4f}).

3. PIB PER CAPITA POR UF
   • Mediana nacional:   R$ {:,.0f}
   • Média (puxada por DF/SP): R$ {:,.0f}
   • Forte assimetria: CV = {:.1f}%

4. RELAÇÃO POPULAÇÃO × PIB PER CAPITA
   • Correlação log-linear: r = {:+.3f}
   • R² do modelo:          {:.3f}
   • Resíduos sugerem heterogeneidade regional não capturada
     apenas pelo tamanho da população.

═══════════════════════════════════════════════════════════
""".format(
    ipca["variacao_mensal"].mean(), ipca["variacao_mensal"].std(),
    antes.mean(), depois.mean(),
    "significativa" if p < 0.05 else "não significativa", p,
    uf["pib_per_capita"].median(), uf["pib_per_capita"].mean(),
    uf["pib_per_capita"].std()/uf["pib_per_capita"].mean()*100,
    stats.pearsonr(uf["log_pop"], uf["pib_per_capita"])[0],
    modelo.rsquared
))

## 10. Próximos passos sugeridos

Depois deste curso, você está pronto para:

1. **Regressão múltipla** — mais de uma variável explicativa.
2. **Séries temporais** — ARIMA, decomposição sazonal.
3. **Modelos lineares generalizados** — Poisson, logística.
4. **Machine Learning estatístico** — começando por regressão regularizada.
5. **Inferência bayesiana** — alternativa moderna ao p-valor.

---

## Encerramento

Você passou de **coletar dados de uma API real** até **construir modelos e
relatórios interativos**, sempre com:
- Teoria explicada antes do código;
- Dados oficiais brasileiros (IBGE/SIDRA);
- Reproducibilidade total (basta clonar o repo e rodar).

Bons estudos e bons dados! 📊